
# XGBoost Nutrient-Aware Pathogen Suppression Predictor 0103

This is the strictest version of the nutrient-aware model.

Features used (41 total):
  - R0_c1  ... R0_c20   : initial nutrient concentrations (continuous)
  - active_c1 ... active_c20 : binary flags — is this nutrient present?
  - diversity_level      : how many carbon sources are active (1/2/4/8/16)

Features intentionally excluded:
  - Species_0002 ... Species_0101 : ALL species membership columns REMOVED
  - community_size                : REMOVED 



Input:  community_index_v8_backup.csv
Target: community_suppression (float in [0, 1])


In [13]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

In [14]:
df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\community_index_v8_backup.csv")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Target — mean: {df['community_suppression'].mean():.3f}  "
      f"std: {df['community_suppression'].std():.3f}  "
      f"range: [{df['community_suppression'].min():.3f}, {df['community_suppression'].max():.3f}]")

Dataset shape: 30,000 rows × 168 columns
Target — mean: 0.654  std: 0.321  range: [0.000, 0.999]


In [15]:
# feature matrix

r0_cols = [f"R0_c{k}" for k in range(1, 21)]

active_cols = [f"active_c{k}" for k in range(1, 21)]

diversity_col = ['diversity_level']


feature_cols = r0_cols + active_cols + diversity_col

print(f"Feature count:          {len(feature_cols)}  (vs 42 in nutrient+size, 142 in full model)")
print(f"  R0 concentrations:    {len(r0_cols)}")
print(f"  Active flags:         {len(active_cols)}")
print(f"  diversity_level:      {len(diversity_col)}")
print(f"  community_size:       0  ← removed (biological, not chemical)")
print(f"  Species membership:   0  ← removed")

# Diagnostic: confirm all feature columns exist in the loaded file
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    print(f"\nWARNING: {len(missing)} missing columns: {missing}")
else:
    print("\nAll feature columns confirmed present in file.")

X = df[feature_cols]
y = df['community_suppression']

print(f"\nX shape: {X.shape}    y shape: {y.shape}")

Feature count:          41  (vs 42 in nutrient+size, 142 in full model)
  R0 concentrations:    20
  Active flags:         20
  diversity_level:      1
  community_size:       0  ← removed (biological, not chemical)
  Species membership:   0  ← removed

All feature columns confirmed present in file.

X shape: (30000, 41)    y shape: (30000,)


In [16]:
# train test split

strat_key = (df['community_size'].astype(str) + "_" +
             df['diversity_level'].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key
)

print(f"Train: {X_train.shape[0]:,} rows   Test: {X_test.shape[0]:,} rows")

Train: 24,000 rows   Test: 6,000 rows


In [17]:
# baseline models

y_pred_mean = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
print(f"Global mean predictor: R² = {r2_score(y_test, y_pred_mean):.4f}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
ridge = Ridge(alpha=1.0).fit(X_train_sc, y_train)
y_pred_ridge = ridge.predict(X_test_sc)
r2_ridge  = r2_score(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
print(f"Ridge Regression:      R² = {r2_ridge:.4f}   MAE = {mae_ridge:.4f}")

Global mean predictor: R² = -0.0003
Ridge Regression:      R² = 0.6812   MAE = 0.1282


In [18]:
# xgboost initial

# With only 41 features and a harder prediction task (no size signal),
# we allow deeper trees (max_depth=8) so the model can still capture
# complex nutrient interaction patterns.
model_init = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_lambda=1.0,
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    verbosity=0
)

model_init.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_init = model_init.predict(X_test)
print(f"XGBoost initial:  R² = {r2_score(y_test, y_pred_init):.4f}   "
      f"MAE = {mean_absolute_error(y_test, y_pred_init):.4f}   "
      f"best iteration: {model_init.best_iteration}")

XGBoost initial:  R² = 0.6812   MAE = 0.1283   best iteration: 161


In [19]:
# hyperparameter tuning


param_grid = {
    'max_depth':        [4,5,6, 8, 10],
    'learning_rate':    [0.01,0.05,0.08,0.1],
    'subsample':        [0.6,0.8,1.0],
    'colsample_bytree': [0.6,0.8,1.0],
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)

base_est = xgb.XGBRegressor(
    n_estimators=200,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    verbosity=0
)

grid_search = GridSearchCV(
    estimator=base_est,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)
grid_search.fit(X_train, y_train)

print(f"\nBest parameters:")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

Fitting 3 folds for each of 180 candidates, totalling 540 fits

Best parameters:
  colsample_bytree: 1.0
  learning_rate: 0.05
  max_depth: 4
  subsample: 1.0
Best CV RMSE: 0.1831


In [20]:
# final model

print(grid_search.best_params_)

model_final = xgb.XGBRegressor(
    **grid_search.best_params_,
    n_estimators=1000,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbosity=0
)

model_final.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

y_pred_final = model_final.predict(X_test)

r2_final   = r2_score(y_test, y_pred_final)
mae_final  = mean_absolute_error(y_test, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))

print(f"\n--- Pure Nutrient Model Performance ---")
print(f"R²:   {r2_final:.4f}")
print(f"MAE:  {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")
print(f"Trees used: {model_final.best_iteration}")

{'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 4, 'subsample': 1.0}
[0]	validation_0-rmse:0.31362
[50]	validation_0-rmse:0.18508
[100]	validation_0-rmse:0.18294
[150]	validation_0-rmse:0.18289
[200]	validation_0-rmse:0.18289
[235]	validation_0-rmse:0.18289

--- Pure Nutrient Model Performance ---
R²:   0.6812
MAE:  0.1283
RMSE: 0.1829
Trees used: 185


In [21]:
# per condition breakdown



results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          y_pred_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

# NOTE: we broke down by community_size here even though it wasn't a feature.
# This is purely for interpretation, we want to see whether the model's
# errors are systematically larger for small vs large communities, which
# would confirm that size carries information the model is now missing.
print("\nBy diversity level:")
print(f"{'D':<5} {'R²':>8}  {'MAE':>8}  {'N':>6}")
print("-" * 32)
for d, grp in results_df.groupby('diversity_level'):
    print(f"D={d:<4} {r2_score(grp.y_true, grp.y_pred):>8.4f}  "
          f"{mean_absolute_error(grp.y_true, grp.y_pred):>8.4f}  {len(grp):>6,}")

print("\nBy community size (not a feature — checking for systematic bias):")
print(f"{'N':<5} {'R²':>8}  {'MAE':>8}  {'Count':>6}")
print("-" * 32)
for n, grp in results_df.groupby('community_size'):
    print(f"N={n:<4} {r2_score(grp.y_true, grp.y_pred):>8.4f}  "
          f"{mean_absolute_error(grp.y_true, grp.y_pred):>8.4f}  {len(grp):>6,}")


By diversity level:
D           R²       MAE       N
--------------------------------
D=1      0.9037    0.0699   1,200
D=2      0.2913    0.2052   1,200
D=4      0.2527    0.1455   1,200
D=8      0.8152    0.0951   1,200
D=16     0.2541    0.1257   1,200

By community size (not a feature — checking for systematic bias):
N           R²       MAE   Count
--------------------------------
N=12     0.4314    0.1826   1,200
N=16     0.6755    0.1298   1,200
N=24     0.8003    0.0958   1,200
N=32     0.7639    0.1049   1,200
N=48     0.6621    0.1284   1,200


In [22]:
# shap analysis

print('shap analysis')

explainer = shap.TreeExplainer(model_final)
rng = np.random.default_rng(42)
idx = rng.choice(len(X_test), size=2000, replace=False)
shap_values = explainer.shap_values(X_test.iloc[idx])

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)

print("\nAll 41 features ranked by mean |SHAP|:")
print(shap_importance.round(5).to_string())

r0_shap     = shap_importance[r0_cols].sum()
active_shap = shap_importance[active_cols].sum()
div_shap    = shap_importance[diversity_col].sum()
total_shap  = r0_shap + active_shap + div_shap

print(f"\nSHAP by group:")
print(f"  R0 concentrations:  {r0_shap:.4f}  ({100*r0_shap/total_shap:.1f}%)")
print(f"  Active flags:       {active_shap:.4f}  ({100*active_shap/total_shap:.1f}%)")
print(f"  diversity_level:    {div_shap:.4f}  ({100*div_shap/total_shap:.1f}%)")

shap analysis

All 41 features ranked by mean |SHAP|:
R0_c8              0.13619
R0_c1              0.08459
R0_c6              0.02908
R0_c12             0.01935
R0_c14             0.01403
R0_c19             0.01355
R0_c3              0.01267
R0_c9              0.01255
R0_c2              0.01107
R0_c15             0.01016
R0_c16             0.00369
R0_c5              0.00363
R0_c11             0.00353
R0_c7              0.00314
R0_c13             0.00078
R0_c18             0.00075
R0_c20             0.00030
diversity_level    0.00007
R0_c4              0.00003
R0_c17             0.00000
R0_c10             0.00000
active_c2          0.00000
active_c3          0.00000
active_c4          0.00000
active_c1          0.00000
active_c5          0.00000
active_c6          0.00000
active_c8          0.00000
active_c7          0.00000
active_c10         0.00000
active_c11         0.00000
active_c12         0.00000
active_c9          0.00000
active_c13         0.00000
active_c14         0.00000
a

In [23]:
# figures


print("\n" + "=" * 60)
print("SECTION 11: Generating plots → pure_nutrient_results.png")
print("=" * 60)

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(
    f'XGBoost PURE NUTRIENT (Chemistry-Only) Suppression Predictor — MiCRM v8\n'
    f'R²={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}  '
    f'[nutrient+size R²≈?  |  full model R²=0.930]',
    fontsize=12, fontweight='bold', y=0.99
)

# Panel 1: Predicted vs Actual coloured by diversity level
ax1 = fig.add_subplot(2, 3, 1)
for d, col in zip([1,2,4,8,16], ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3']):
    mask = df.loc[X_test.index, 'diversity_level'].values == d
    ax1.scatter(y_test.values[mask], y_pred_final[mask],
                alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.plot([0,1],[0,1],'r--',lw=1.5)
ax1.set_xlabel('True suppression')
ax1.set_ylabel('Predicted suppression')
ax1.set_title(f'Predicted vs Actual  R²={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)

# Panel 2: Residuals coloured by community size
# Even though size wasn't a feature, colouring by it reveals whether
# the model is systematically biased against certain community sizes.
ax2 = fig.add_subplot(2, 3, 2)
resid = y_test.values - y_pred_final
for n, col in zip([12,16,24,32,48], ['#1b9e77','#d95f02','#7570b3','#e7298a','#66a61e']):
    mask = df.loc[X_test.index, 'community_size'].values == n
    ax2.scatter(y_pred_final[mask], resid[mask],
                alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted suppression')
ax2.set_ylabel('Residual (true − predicted)')
ax2.set_title('Residuals (coloured by size — not a feature)')
ax2.legend(fontsize=7, markerscale=3)

# Panel 3: SHAP importance — all 41 features
ax3 = fig.add_subplot(2, 3, 3)
def feat_colour(f):
    if f in r0_cols:     return '#4CAF50'
    if f in active_cols: return '#FF9800'
    return '#9C27B0'   # diversity_level
top_n   = len(feature_cols)
top_shap = shap_importance.head(top_n)
cols = [feat_colour(f) for f in top_shap.index]
ax3.barh(range(top_n), top_shap.values[::-1], color=cols[::-1])
ax3.set_yticks(range(top_n))
ax3.set_yticklabels(top_shap.index[::-1], fontsize=6)
ax3.set_xlabel('Mean |SHAP value|')
ax3.set_title('All 41 Features by SHAP')
ax3.legend(handles=[
    Patch(fc='#4CAF50', label='R0 concentration'),
    Patch(fc='#FF9800', label='Active flag'),
    Patch(fc='#9C27B0', label='diversity_level'),
], loc='lower right', fontsize=8)

# Panel 4: R² by diversity level
ax4 = fig.add_subplot(2, 3, 4)
divs   = sorted(results_df['diversity_level'].unique())
r2_div = [r2_score(results_df[results_df['diversity_level']==d]['y_true'],
                   results_df[results_df['diversity_level']==d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level (# active nutrients)')
ax4.set_ylabel('R²')
ax4.set_title('R² by Diversity Level')
ax4.set_ylim(0, 1.05)
ax4.legend(fontsize=8)

# Panel 5: R² by community size
# Key diagnostic: if R² varies strongly by size, the model is missing
# the community_size signal that we deliberately excluded.
ax5 = fig.add_subplot(2, 3, 5)
sizes  = sorted(results_df['community_size'].unique())
r2_sz  = [r2_score(results_df[results_df['community_size']==n]['y_true'],
                   results_df[results_df['community_size']==n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N (NOT a feature in this model)')
ax5.set_ylabel('R²')
ax5.set_title('R² by Community Size\n(size was excluded — shows what we lost)')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=8)

# Panel 6: SHAP group pie
ax6 = fig.add_subplot(2, 3, 6)
ax6.pie(
    [r0_shap, active_shap, div_shap],
    labels=['R0\nconcentrations', 'Active\nflags', 'diversity_level'],
    colors=['#4CAF50', '#FF9800', '#9C27B0'],
    autopct='%.1f%%', startangle=90, textprops={'fontsize': 10}
)
ax6.set_title('SHAP Importance by Feature Group\n(pure nutrient model)')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('pure_nutrient_results.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: pure_nutrient_results.png")


SECTION 11: Generating plots → pure_nutrient_results.png
Saved: pure_nutrient_results.png


In [ ]:
# =============================================================================
# SECTION 12 — SAVE OUTPUTS
# =============================================================================

model_final.save_model('xgb_pure_nutrient_v8.json')
pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          y_pred_final,
    'residual':        y_test.values - y_pred_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
}).to_csv('predictions_pure_nutrient_v8.csv', index=False)
print("Saved: xgb_pure_nutrient_v8.json, predictions_pure_nutrient_v8.csv")